# Train the Parking Plate Detector

This notebook fine-tunes YOLO11n on the cleaned license-plate dataset and evaluates the final model once on the held-out test split. Run it in Google Colab with a GPU runtime.

**Before starting:** in Colab, choose **Runtime → Change runtime type → T4 GPU** (or another available GPU). The cleaned dataset ZIP is about 422 MB. Keep the test split untouched until the final evaluation cell.


In [ ]:
# Install Ultralytics in this Colab runtime.
%pip -q install ultralytics


In [ ]:
# Confirm the runtime has a GPU before training.
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected. In Colab, select Runtime > Change runtime type > GPU, then rerun this cell.")
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Upload Data Set Cleaned.zip from your computer.
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if not zip_names:
    raise ValueError("Upload the Data Set Cleaned.zip file.")
zip_path = Path(zip_names[0])
data_root = Path("/content/Data Set Cleaned")
data_root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(data_root)

config_path = data_root / "data_colab.yaml"
assert config_path.is_file(), f"Missing config: {config_path}"
for split in ("train", "val", "test"):
    assert (data_root / split / "images").is_dir(), f"Missing {split} images"
    assert (data_root / split / "labels").is_dir(), f"Missing {split} labels"
print("Dataset extracted to:", data_root)
print(config_path.read_text())


## Train

This starts from pretrained YOLO11n weights. The settings use a small batch size to reduce GPU memory use. If Colab reports an out-of-memory error, lower `batch` from 8 to 4 and rerun training.


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
train_result = model.train(
    data=str(config_path),
    epochs=30,
    imgsz=640,
    batch=8,
    patience=8,
    workers=2,
    device=0,
    project="/content/parking_runs",
    name="plate_detector_yolo11n",
    exist_ok=True,
    seed=42,
)

best_weights = Path(train_result.save_dir) / "weights" / "best.pt"
assert best_weights.is_file(), f"Training weights not found: {best_weights}"
print("Best weights:", best_weights)
print("Run outputs:", train_result.save_dir)


## Final test evaluation

The validation split is used during training to select the best checkpoint. This next cell evaluates that checkpoint on the separate test split for the final reported detection metrics.


In [ ]:
best_model = YOLO(str(best_weights))
test_metrics = best_model.val(
    data=str(config_path),
    split="test",
    imgsz=640,
    batch=8,
    device=0,
    plots=True,
    project="/content/parking_runs",
    name="plate_detector_test",
    exist_ok=True,
)
print("Held-out test metrics:")
for key, value in test_metrics.results_dict.items():
    print(f"{key}: {value}")


## Inspect a few test predictions

Look at the saved prediction images and confirm that boxes surround the plate itself. Record any misses or incorrect boxes for the report.


In [ ]:
test_images = data_root / "test" / "images"
predictions = best_model.predict(
    source=str(test_images),
    imgsz=640,
    conf=0.25,
    device=0,
    save=True,
    project="/content/parking_runs",
    name="test_predictions",
    exist_ok=True,
)
print("Prediction images:", "/content/parking_runs/test_predictions")


In [ ]:
# Download the trained model to your computer for the local demo.
files.download(str(best_weights))
